In [1]:
import json

d = json.load(open(r"C:\Users\LENOVO\Documents\Datasets\flir_align\coco_annotations\train_new.json"))

print("Top-level keys:", list(d.keys()))
print("Full sample image entry:", d['images'][0])

# check if ANY image entry has extra fields beyond id/file_name/width/height
extra_keys = set()
for img in d['images']:
    extra_keys.update(img.keys())
print("All keys seen across every image entry:", extra_keys)

Top-level keys: ['images', 'type', 'annotations', 'categories']
Full sample image entry: {'id': 'FLIR_00258', 'file_name': 'FLIR_00258.jpeg', 'width': 640, 'height': 512}
All keys seen across every image entry: {'id', 'width', 'file_name', 'height'}


In [2]:
import cv2
import numpy as np
from pathlib import Path

VAL_DIR = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo\images\val")

brightnesses = []
for img_path in sorted(VAL_DIR.glob("*.jpeg")):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    brightnesses.append(img.mean())

brightnesses = np.array(brightnesses)
print(f"n images: {len(brightnesses)}")
print(f"min: {brightnesses.min():.1f}  max: {brightnesses.max():.1f}  mean: {brightnesses.mean():.1f}")
print("Percentiles:")
for p in [10, 25, 40, 50, 60, 75, 90]:
    print(f"  {p}th: {np.percentile(brightnesses, p):.1f}")

n images: 1013
min: 74.2  max: 205.9  mean: 166.3
Percentiles:
  10th: 115.5
  25th: 132.7
  40th: 175.1
  50th: 179.9
  60th: 185.6
  75th: 191.7
  90th: 196.0


In [3]:
import cv2
import numpy as np
from pathlib import Path

VAL_DIR = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo\images\val")

brightnesses = []
for img_path in sorted(VAL_DIR.glob("*.jpeg")):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    brightnesses.append(img.mean())

brightnesses = np.array(brightnesses)

# fine-grained histogram just in the gap region
counts, bin_edges = np.histogram(brightnesses, bins=np.arange(100, 200, 5))
for i, c in enumerate(counts):
    print(f"{bin_edges[i]:.0f}-{bin_edges[i+1]:.0f}: {c}")

100-105: 17
105-110: 13
110-115: 39
115-120: 72
120-125: 45
125-130: 22
130-135: 38
135-140: 12
140-145: 12
145-150: 17
150-155: 7
155-160: 4
160-165: 10
165-170: 12
170-175: 63
175-180: 104
180-185: 95
185-190: 122
190-195: 160


In [4]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\LENOVO\Documents\Datasets\FLIR_ADAS_v2\images_rgb_train")

# Check coco.json first
coco_path = ROOT / "coco.json"
if coco_path.exists():
    d = json.load(open(coco_path))
    print("=== coco.json ===")
    print("Top-level keys:", list(d.keys()))
    print("Sample image entry:", d['images'][0])
    extra_keys = set()
    for img in d['images']:
        extra_keys.update(img.keys())
    print("All keys across all image entries:", extra_keys)
else:
    print(f"coco.json not found at {coco_path}")

print()

# Check index.json -- Teledyne's own richer format, may carry more metadata
index_path = ROOT / "index.json"
if index_path.exists():
    d = json.load(open(index_path))
    print("=== index.json ===")
    print("Type:", type(d))
    if isinstance(d, dict):
        print("Top-level keys:", list(d.keys()))
    elif isinstance(d, list):
        print("First entry:", d[0])
else:
    print(f"index.json not found at {index_path}")

=== coco.json ===
Top-level keys: ['annotations', 'categories', 'images', 'info', 'licenses']
Sample image entry: {'extra_info': {'hours': 'night', 'scene': 'other', 'video_id': '23bsd9bsr962GdFBZ'}, 'file_name': 'data/video-23bsd9bsr962GdFBZ-frame-000221-arDqqfMYcrbEEqsex.jpg', 'height': 1600, 'id': 0, 'subdirs': 'data', 'width': 1800}
All keys across all image entries: {'subdirs', 'extra_info', 'file_name', 'height', 'id', 'width'}

=== index.json ===
Type: <class 'dict'>
Top-level keys: ['datasetId', 'datasetName', 'frames', 'owner', 'version', 'videos']


In [5]:
import cv2
import json
import numpy as np
from pathlib import Path

VAL_DIR = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo\images\val")
OUT_PATH = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo\val_day_night_labels.json")

THRESHOLD = 157  # mean grayscale brightness; below = night, at/above = day

labels = {}
for img_path in sorted(VAL_DIR.glob("*.jpeg")):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    brightness = float(img.mean())
    labels[img_path.stem] = "night" if brightness < THRESHOLD else "day"

n_night = sum(1 for v in labels.values() if v == "night")
n_day = sum(1 for v in labels.values() if v == "day")
print(f"night: {n_night} ({100*n_night/len(labels):.1f}%)")
print(f"day: {n_day} ({100*n_day/len(labels):.1f}%)")

with open(OUT_PATH, "w") as f:
    json.dump(labels, f, indent=2)
print(f"Saved to {OUT_PATH}")

night: 316 (31.2%)
day: 697 (68.8%)
Saved to C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo\val_day_night_labels.json


In [6]:
from ultralytics import YOLO
import torch

model = YOLO("yolov8n.pt")
backbone = model.model.model  # the underlying nn.Sequential of layers

dummy_input = torch.randn(1, 3, 640, 640)

x = dummy_input
for i, layer in enumerate(backbone):
    x = layer(x) if not isinstance(x, list) else layer(x)
    shape = x.shape if torch.is_tensor(x) else [t.shape for t in x]
    print(f"layer {i}: {type(layer).__name__:20s} -> {shape}")
    if i >= 10:  # backbone is roughly the first ~10 layers; head starts after
        break

layer 0: Conv                 -> torch.Size([1, 16, 320, 320])
layer 1: Conv                 -> torch.Size([1, 32, 160, 160])
layer 2: C2f                  -> torch.Size([1, 32, 160, 160])
layer 3: Conv                 -> torch.Size([1, 64, 80, 80])
layer 4: C2f                  -> torch.Size([1, 64, 80, 80])
layer 5: Conv                 -> torch.Size([1, 128, 40, 40])
layer 6: C2f                  -> torch.Size([1, 128, 40, 40])
layer 7: Conv                 -> torch.Size([1, 256, 20, 20])
layer 8: C2f                  -> torch.Size([1, 256, 20, 20])
layer 9: SPPF                 -> torch.Size([1, 256, 20, 20])
layer 10: Upsample             -> torch.Size([1, 256, 40, 40])


In [1]:
import json
from pathlib import Path

RAW_ROOT = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_align")

for split_name, json_name in [("train", "train_new.json"), ("val", "test_new.json")]:
    src_path = RAW_ROOT / "coco_annotations" / json_name
    with open(src_path) as f:
        data = json.load(f)

    # build old string id -> new sequential int id mapping
    id_map = {img["id"]: i for i, img in enumerate(data["images"])}

    for img in data["images"]:
        img["id"] = id_map[img["id"]]

    for ann in data["annotations"]:
        ann["image_id"] = id_map[ann["image_id"]]

    out_name = json_name.replace(".json", "_intid.json")
    out_path = RAW_ROOT / "coco_annotations" / out_name
    with open(out_path, "w") as f:
        json.dump(data, f)

    print(f"{split_name}: wrote {out_path} ({len(data['images'])} images, {len(data['annotations'])} annotations)")

train: wrote C:\Users\LENOVO\Documents\Datasets\flir_align\coco_annotations\train_new_intid.json (4129 images, 32161 annotations)
val: wrote C:\Users\LENOVO\Documents\Datasets\flir_align\coco_annotations\test_new_intid.json (1013 images, 8591 annotations)


In [2]:
import json
from pathlib import Path

RAW_ROOT = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_align")
DAY_NIGHT_LABELS_PATH = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo\val_day_night_labels.json")

with open(DAY_NIGHT_LABELS_PATH) as f:
    day_night_labels = json.load(f)
night_stems = {stem for stem, label in day_night_labels.items() if label == "night"}

with open(RAW_ROOT / "coco_annotations" / "test_new_intid.json") as f:
    data = json.load(f)

# filter by file_name stem, not by id -- ids were remapped to ints earlier,
# but file_name still preserves the original FLIR_XXXXX naming, which is
# what val_day_night_labels.json's keys match against
night_images = [img for img in data["images"] if Path(img["file_name"]).stem in night_stems]
night_image_ids = {img["id"] for img in night_images}

night_annotations = [ann for ann in data["annotations"] if ann["image_id"] in night_image_ids]

night_data = {
    "images": night_images,
    "annotations": night_annotations,
    "categories": data["categories"],
}

out_path = RAW_ROOT / "coco_annotations" / "test_new_intid_night.json"
with open(out_path, "w") as f:
    json.dump(night_data, f)

print(f"wrote {out_path}")
print(f"night images: {len(night_images)}  night annotations: {len(night_annotations)}")

wrote C:\Users\LENOVO\Documents\Datasets\flir_align\coco_annotations\test_new_intid_night.json
night images: 316  night annotations: 2623


In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

ann_dir = Path(r"C:\Users\LENOVO\Documents\Datasets\M3FD_Detection\Annotation")
class_counts = Counter()

for xml_path in ann_dir.glob("*.xml"):
    tree = ET.parse(xml_path)
    for obj in tree.getroot().findall("object"):
        class_counts[obj.find("name").text] += 1

for cls, count in class_counts.most_common():
    print(f"{cls}: {count}")

Car: 18296
People: 11477
Lamp: 2405
Truck: 1008
Bus: 700
Motorcycle: 521


In [2]:
import xml.etree.ElementTree as ET
import json
import random
import shutil
from pathlib import Path

M3FD_ROOT = Path(r"C:\Users\LENOVO\Documents\Datasets\M3FD_Detection")
VIS_DIR = M3FD_ROOT / "Vis"
ANN_DIR = M3FD_ROOT / "Annotation"

OUT_COCO_DIR = M3FD_ROOT / "coco_annotations"
OUT_YOLO_DIR = Path(r"C:\Users\LENOVO\Documents\Datasets\m3fd_yolo")

CLASS_NAMES = ["Bus", "Car", "Lamp", "Motorcycle", "People", "Truck"]  # alphabetical, fixed order
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

VAL_FRACTION = 0.2
SEED = 42

OUT_COCO_DIR.mkdir(parents=True, exist_ok=True)
for split in ("train", "val"):
    (OUT_YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# ---- parse every XML once, build a unified record per image ----
records = []  # list of dicts: {id, file_name, width, height, boxes: [(cls_id, xmin,ymin,xmax,ymax)]}
for xml_path in sorted(ANN_DIR.glob("*.xml")):
    stem = xml_path.stem  # e.g. "00000"
    img_id = int(stem)    # M3FD filenames are already numeric -- no string-id workaround needed

    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    boxes = []
    for obj in root.findall("object"):
        cls_name = obj.find("name").text
        cls_id = CLASS_TO_ID[cls_name]
        bnd = obj.find("bndbox")
        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)
        boxes.append((cls_id, xmin, ymin, xmax, ymax))

    records.append({
        "id": img_id, "file_name": f"{stem}.png",
        "width": width, "height": height, "boxes": boxes,
    })

print(f"parsed {len(records)} annotation files")

# ---- train/val split, fixed seed for reproducibility ----
random.seed(SEED)
shuffled = records.copy()
random.shuffle(shuffled)
n_val = int(len(shuffled) * VAL_FRACTION)
val_records = shuffled[:n_val]
train_records = shuffled[n_val:]
print(f"train: {len(train_records)}  val: {len(val_records)}")


def write_coco(records_subset, out_path):
    images, annotations = [], []
    ann_id = 1
    for rec in records_subset:
        images.append({"id": rec["id"], "file_name": rec["file_name"],
                        "width": rec["width"], "height": rec["height"]})
        for cls_id, xmin, ymin, xmax, ymax in rec["boxes"]:
            annotations.append({
                "id": ann_id, "image_id": rec["id"], "category_id": cls_id,
                "bbox": [xmin, ymin, xmax - xmin, ymax - ymin],
                "area": (xmax - xmin) * (ymax - ymin), "iscrowd": 0,
            })
            ann_id += 1
    categories = [{"id": i, "name": name} for i, name in enumerate(CLASS_NAMES)]
    with open(out_path, "w") as f:
        json.dump({"images": images, "annotations": annotations, "categories": categories}, f)
    print(f"wrote {out_path} ({len(images)} images, {len(annotations)} annotations)")


write_coco(train_records, OUT_COCO_DIR / "train_m3fd.json")
write_coco(val_records, OUT_COCO_DIR / "val_m3fd.json")


def write_yolo(records_subset, split_name):
    img_out = OUT_YOLO_DIR / "images" / split_name
    lbl_out = OUT_YOLO_DIR / "labels" / split_name
    for rec in records_subset:
        src_img = VIS_DIR / rec["file_name"]
        shutil.copy(src_img, img_out / rec["file_name"])

        w, h = rec["width"], rec["height"]
        lines = []
        for cls_id, xmin, ymin, xmax, ymax in rec["boxes"]:
            xc = (xmin + xmax) / 2 / w
            yc = (ymin + ymax) / 2 / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        label_path = lbl_out / (rec["file_name"].replace(".png", ".txt"))
        with open(label_path, "w") as f:
            f.write("\n".join(lines))
    print(f"[{split_name}] wrote {len(records_subset)} YOLO image/label pairs")


write_yolo(train_records, "train")
write_yolo(val_records, "val")

print("\nClass mapping used (keep this consistent everywhere downstream):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name}")

parsed 4200 annotation files
train: 3360  val: 840
wrote C:\Users\LENOVO\Documents\Datasets\M3FD_Detection\coco_annotations\train_m3fd.json (3360 images, 27324 annotations)
wrote C:\Users\LENOVO\Documents\Datasets\M3FD_Detection\coco_annotations\val_m3fd.json (840 images, 7083 annotations)
[train] wrote 3360 YOLO image/label pairs
[val] wrote 840 YOLO image/label pairs

Class mapping used (keep this consistent everywhere downstream):
  0: Bus
  1: Car
  2: Lamp
  3: Motorcycle
  4: People
  5: Truck
